In [ ]:
import cv2
import numpy as np

try:
    from PIL import Image
except Exception:
    Image = None


In [ ]:
#exercice 1 chargement images zebra
def read_gray_safe(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        return img

    # fallback pour certains TIFF 4-bit non lus par OpenCV
    if Image is not None:
        try:
            return np.array(Image.open(path).convert('L'))
        except Exception as e:
            raise AssertionError(f"Erreur lecture image {path}: {e}")

    raise AssertionError(f"Image introuvable ou illisible: {path}")


zebra_files = [f"images/zebra_{i}.tif" for i in range(1, 7)]
zebra_gray = []

for f in zebra_files:
    img = read_gray_safe(f)
    zebra_gray.append(img)

print("zebra images chargees:", len(zebra_gray))


In [ ]:
#ex1.1 glcm maison + features
def quantize_img(gray, levels=16):
    q = gray // (256 // levels)
    q[q >= levels] = levels - 1
    return q


def my_glcm(win_q, levels=16, offsets=((0,1),(1,0),(1,1),(-1,1))):
    h, w = win_q.shape
    P = [[0.0 for _ in range(levels)] for _ in range(levels)]

    for dy, dx in offsets:
        y0 = 0 if dy >= 0 else -dy
        y1 = h - dy if dy >= 0 else h
        x0 = 0 if dx >= 0 else -dx
        x1 = w - dx if dx >= 0 else w

        for y in range(y0, y1):
            for x in range(x0, x1):
                a = int(win_q[y, x])
                b = int(win_q[y + dy, x + dx])
                P[a][b] += 1.0
                P[b][a] += 1.0

    s = 0.0
    for i in range(levels):
        for j in range(levels):
            s += P[i][j]

    if s > 0.0:
        inv = 1.0 / s
        for i in range(levels):
            for j in range(levels):
                P[i][j] *= inv

    return P


def glcm_features(P):
    levels = len(P)

    mu = 0.0
    for i in range(levels):
        for j in range(levels):
            mu += i * P[i][j]

    var_ = 0.0
    contrast = 0.0
    entropy = 0.0

    for i in range(levels):
        for j in range(levels):
            p = P[i][j]
            var_ += ((i - mu) * (i - mu)) * p
            contrast += ((i - j) * (i - j)) * p
            if p > 0.0:
                entropy += -p * (np.log(p + 1e-12) / np.log(2.0))

    return var_, contrast, entropy


def sliding_glcm_maps(gray, win=19, levels=16, step=3):
    pad = win // 2
    q = quantize_img(gray, levels)
    qpad = cv2.copyMakeBorder(q, pad, pad, pad, pad, cv2.BORDER_REFLECT)

    h, w = gray.shape
    var_map = gray.astype('float32')
    con_map = gray.astype('float32')
    ent_map = gray.astype('float32')

    var_map[:] = 0.0
    con_map[:] = 0.0
    ent_map[:] = 0.0

    for y in range(0, h, step):
        for x in range(0, w, step):
            block = qpad[y:y+win, x:x+win]
            P = my_glcm(block, levels)
            v, c, e = glcm_features(P)

            y2 = y + step
            x2 = x + step
            if y2 > h:
                y2 = h
            if x2 > w:
                x2 = w

            var_map[y:y2, x:x2] = v
            con_map[y:y2, x:x2] = c
            ent_map[y:y2, x:x2] = e

    return var_map, con_map, ent_map


def to_u8(im):
    return cv2.normalize(im, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')


In [ ]:
#ex1.2 thresholding texture mask
img = zebra_gray[0]
var_map, con_map, ent_map = sliding_glcm_maps(img, win=19, levels=16, step=3)

v8 = to_u8(var_map)
c8 = to_u8(con_map)
e8 = to_u8(ent_map)

_, mv = cv2.threshold(v8, 155, 255, cv2.THRESH_BINARY)
_, mc = cv2.threshold(c8, 165, 255, cv2.THRESH_BINARY)
_, me = cv2.threshold(e8, 165, 255, cv2.THRESH_BINARY)

mask = cv2.bitwise_and(mv, mc)
mask = cv2.bitwise_and(mask, me)

k3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
k5 = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k3)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k5)

img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
mask_color = cv2.applyColorMap(mask, cv2.COLORMAP_SPRING)
overlay = cv2.addWeighted(img_bgr, 0.75, mask_color, 0.35, 0)

line1 = cv2.hconcat([img_bgr, cv2.applyColorMap(v8, cv2.COLORMAP_INFERNO), cv2.applyColorMap(c8, cv2.COLORMAP_INFERNO)])
line2 = cv2.hconcat([cv2.applyColorMap(e8, cv2.COLORMAP_INFERNO), cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR), overlay])
out = cv2.vconcat([line1, line2])

cv2.imwrite('images/tp8_ex1_glcm.png', out)
print('saved: images/tp8_ex1_glcm.png')


In [ ]:
#ex1.2 boucle sur zebra_1 ... zebra_6
for i, img in enumerate(zebra_gray):
    var_map, con_map, ent_map = sliding_glcm_maps(img, win=19, levels=16, step=4)

    v8 = to_u8(var_map)
    c8 = to_u8(con_map)
    e8 = to_u8(ent_map)

    _, mv = cv2.threshold(v8, 155, 255, cv2.THRESH_BINARY)
    _, mc = cv2.threshold(c8, 165, 255, cv2.THRESH_BINARY)
    _, me = cv2.threshold(e8, 165, 255, cv2.THRESH_BINARY)

    mask = cv2.bitwise_and(cv2.bitwise_and(mv, mc), me)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (3,3)))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (5,5)))

    img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    overlay = cv2.addWeighted(img_bgr, 0.75, cv2.applyColorMap(mask, cv2.COLORMAP_SPRING), 0.35, 0)

    out_name = f'images/tp8_zebra_mask_{i+1}.png'
    cv2.imwrite(out_name, overlay)
    print('saved:', out_name)


In [ ]:
#ex1.3 comparaison avec fonctions de bibliotheque (variance + entropie)
img = zebra_gray[0]
img_f = img.astype('float32')
ksize = 19

# version opencv
mean = cv2.blur(img_f, (ksize, ksize))
mean2 = cv2.blur(cv2.multiply(img_f, img_f), (ksize, ksize))
var_cv = cv2.subtract(mean2, cv2.multiply(mean, mean))

q = quantize_img(img, levels=16)
ent_cv = img_f.copy()
ent_cv[:] = 0.0

for b in range(16):
    mb = cv2.inRange(q, b, b)
    p = cv2.multiply(mb.astype('float32'), 1.0 / 255.0)
    pb = cv2.blur(p, (ksize, ksize))
    logpb = cv2.log(cv2.add(pb, 1e-12))
    term = cv2.multiply(pb, logpb)
    term = cv2.multiply(term, -1.0 / 0.69314718056)
    ent_cv = cv2.add(ent_cv, term)

# version bibliotheque
lib_ok = True
try:
    import numpy as np
    from scipy import ndimage as ndi
    from skimage.filters.rank import entropy
    from skimage.morphology import square
except Exception as e:
    lib_ok = False
    print('scipy/skimage manquants:', e)

if lib_ok:
    var_lib = ndi.generic_filter(img.astype(np.float32), np.var, size=ksize)
    ent_lib = entropy(img, square(ksize)).astype('float32')

    cmp_img = cv2.hconcat([
        cv2.cvtColor(img, cv2.COLOR_GRAY2BGR),
        cv2.applyColorMap(to_u8(var_cv), cv2.COLORMAP_INFERNO),
        cv2.applyColorMap(to_u8(var_lib), cv2.COLORMAP_INFERNO),
        cv2.applyColorMap(to_u8(ent_cv), cv2.COLORMAP_INFERNO),
        cv2.applyColorMap(to_u8(ent_lib), cv2.COLORMAP_INFERNO)
    ])

    cv2.imwrite('images/tp8_ex1_first_order_compare.png', cmp_img)
    print('saved: images/tp8_ex1_first_order_compare.png')
else:
    cmp_img = cv2.hconcat([
        cv2.cvtColor(img, cv2.COLOR_GRAY2BGR),
        cv2.applyColorMap(to_u8(var_cv), cv2.COLORMAP_INFERNO),
        cv2.applyColorMap(to_u8(ent_cv), cv2.COLORMAP_INFERNO)
    ])

    cv2.imwrite('images/tp8_ex1_first_order_compare.png', cmp_img)
    print('saved: images/tp8_ex1_first_order_compare.png (opencv seulement)')


In [ ]:
#ex1.4 comparaison LBP maison vs LBP bibliotheque
img = zebra_gray[0]
h, w = img.shape
lbp = img.copy()
lbp[:] = 0

for y in range(1, h-1):
    for x in range(1, w-1):
        c = int(img[y, x])
        code = 0

        if int(img[y-1, x-1]) >= c: code |= 1 << 0
        if int(img[y-1, x  ]) >= c: code |= 1 << 1
        if int(img[y-1, x+1]) >= c: code |= 1 << 2
        if int(img[y  , x+1]) >= c: code |= 1 << 3
        if int(img[y+1, x+1]) >= c: code |= 1 << 4
        if int(img[y+1, x  ]) >= c: code |= 1 << 5
        if int(img[y+1, x-1]) >= c: code |= 1 << 6
        if int(img[y  , x-1]) >= c: code |= 1 << 7

        lbp[y, x] = code

lib_ok = True
try:
    from skimage.feature import local_binary_pattern
except Exception as e:
    lib_ok = False
    print('skimage manquant:', e)

if lib_ok:
    lbp_lib = local_binary_pattern(img, P=8, R=1, method='default')
    lbp_lib_u8 = to_u8(lbp_lib.astype('float32'))

    lbp_u8 = to_u8(lbp.astype('float32'))
    diff = cv2.absdiff(lbp_u8, lbp_lib_u8)

    out = cv2.hconcat([
        cv2.cvtColor(img, cv2.COLOR_GRAY2BGR),
        cv2.applyColorMap(lbp_u8, cv2.COLORMAP_TURBO),
        cv2.applyColorMap(lbp_lib_u8, cv2.COLORMAP_TURBO),
        cv2.applyColorMap(diff, cv2.COLORMAP_INFERNO)
    ])

    cv2.imwrite('images/tp8_ex1_lbp_compare.png', out)
    print('saved: images/tp8_ex1_lbp_compare.png')
else:
    out = cv2.hconcat([
        cv2.cvtColor(img, cv2.COLOR_GRAY2BGR),
        cv2.applyColorMap(lbp, cv2.COLORMAP_TURBO)
    ])

    cv2.imwrite('images/tp8_ex1_lbp_compare.png', out)
    print('saved: images/tp8_ex1_lbp_compare.png (lbp maison seulement)')


In [ ]:
#exercice 2 feature matching (ORB + BFMatcher)
img1 = zebra_gray[0]
img2 = zebra_gray[1]

orb = cv2.ORB_create(nfeatures=1500)
kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
knn = bf.knnMatch(des1, des2, k=2)

good = []
for pair in knn:
    if len(pair) < 2:
        continue
    m = pair[0]
    n = pair[1]
    if m.distance < 0.75 * n.distance:
        good.append(m)

good = sorted(good, key=lambda x: x.distance)

print('keypoints image1:', len(kp1))
print('keypoints image2:', len(kp2))
print('good matches:', len(good))

match_img = cv2.drawMatches(
    img1, kp1,
    img2, kp2,
    good[:80], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

cv2.imwrite('images/tp8_ex2_matching.png', match_img)
print('saved: images/tp8_ex2_matching.png')


In [ ]:
#exercice 3.1 label image
coins = cv2.imread('images/coins.png', cv2.IMREAD_GRAYSCALE)
assert coins is not None, 'Image introuvable: images/coins.png'

_, bin_a = cv2.threshold(coins, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
_, bin_b = cv2.threshold(coins, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

fg_a = cv2.countNonZero(bin_a) / float(bin_a.shape[0] * bin_a.shape[1])
fg_b = cv2.countNonZero(bin_b) / float(bin_b.shape[0] * bin_b.shape[1])

if 0.02 < fg_a < 0.70:
    binary = bin_a
else:
    binary = bin_b

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
print('nombre objets:', num_labels - 1)

label_u8 = cv2.normalize(labels.astype('float32'), None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
label_color = cv2.applyColorMap(label_u8, cv2.COLORMAP_JET)

line = cv2.hconcat([
    cv2.cvtColor(coins, cv2.COLOR_GRAY2BGR),
    cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR),
    label_color
])

cv2.imwrite('images/tp8_ex3_labels.png', line)
print('saved: images/tp8_ex3_labels.png')


In [ ]:
#exercice 3.2 3.3 3.4 moments aire centroide orientation
vis = cv2.cvtColor(coins, cv2.COLOR_GRAY2BGR)
areas = []
angles = []

for lab in range(1, num_labels):
    area = int(stats[lab, cv2.CC_STAT_AREA])
    if area < 20:
        continue

    obj = cv2.inRange(labels, lab, lab)
    m = cv2.moments(obj, binaryImage=True)

    if m['m00'] == 0:
        continue

    cx = int(m['m10'] / m['m00'])
    cy = int(m['m01'] / m['m00'])

    ang = 0.5 * cv2.fastAtan2(2.0 * m['mu11'], (m['mu20'] - m['mu02']))

    areas.append(float(area))
    angles.append(float(ang))

    cv2.circle(vis, (cx, cy), 2, (0, 0, 255), -1)
    cv2.putText(vis, f"{ang:.1f}", (cx + 4, cy - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 0), 1, cv2.LINE_AA)

if len(areas) > 0:
    a_min = min(areas)
    a_max = max(areas)
    a_mean = sum(areas) / len(areas)

    ang_mean = sum(angles) / len(angles)
    ang_var = 0.0
    for x in angles:
        ang_var += (x - ang_mean) * (x - ang_mean)
    ang_std = (ang_var / len(angles)) ** 0.5

    print('objets valides:', len(areas))
    print('aire min/mean/max:', a_min, a_mean, a_max)
    print('orientation mean/std:', ang_mean, ang_std)

cv2.imwrite('images/tp8_ex3_moments.png', vis)
print('saved: images/tp8_ex3_moments.png')
